# Problematic-Rejected PRs — Exploration

**Goal:** Identify and characterise AI-generated pull requests from the AIDev dataset that were **closed without being merged** — i.e. the AI's contribution was outright rejected.

---

## What has been done so far

### 1. Data loading
Five tables are pulled from the `hao-li/AIDev` HuggingFace dataset:
- `pull_request` — PR metadata (title, agent, repo, created/closed/merged timestamps)
- `pr_commits` — commits attached to each PR
- `pr_reviews` — formal review events (APPROVED, CHANGES_REQUESTED, DISMISSED, COMMENTED)
- `pr_review_comments_v2` — inline review comments tied to specific diff hunks
- `pr_timeline` — full event timeline per PR

### 2. Isolating rejected PRs
A PR is considered **rejected** if:
- `merged_at` is null (never merged), AND
- `closed_at` is non-null (was explicitly closed)

The `categorize()` helper from `helpers.py` is then applied to each PR to classify its review history into one of: `changes_requested_no_approval`, `changes_requested_then_approved`, `approved_only`, `commented_only`, `no_reviews`, etc.

### 3. Filter placeholder (cell 3)
A stub cell is left to narrow `rejected` down to the specific `review_category` values that define *problematic-rejected*. The full category distribution is printed in cell 2 to guide that decision.

### 4. Bot noise removal
PRs where every `CHANGES_REQUESTED` review was submitted by a bot (`user_type != 'User'`) are excluded, ensuring the rejection signal comes from a human reviewer.

### 5. Supporting subsets
Three filtered DataFrames are built over the final `rejected_prs` set:
- `pr_reviews_sub` — reviews on these PRs only
- `pr_commits_sub` — commits on these PRs only
- `cr_inline_comments` — inline comments from human `CHANGES_REQUESTED` reviews, with `pr_id` joined back in

### 6. Summary + spot-check
A summary cell prints total counts and category breakdown. A spot-check cell lets you inspect one PR end-to-end (reviews, commits, inline comments).

---

## Next steps
- Fill in cell 3 with the `review_category` values that define your *problematic-rejected* cohort.
- Add any additional filters (e.g. minimum commit count, specific agents, date range).
- Deep-dive analysis cells go below the spot-check cell.

In [1]:
import pandas as pd
from huggingface_hub import hf_hub_download

def load_table(filename):
    path = hf_hub_download(repo_id="hao-li/AIDev", filename=filename, repo_type="dataset")
    return pd.read_parquet(path)

prs      = load_table("pull_request.parquet")
commits  = load_table("pr_commits.parquet")
reviews  = load_table("pr_reviews.parquet")
rev_cmts = load_table("pr_review_comments_v2.parquet")
timeline = load_table("pr_timeline.parquet")
details  = load_table("pr_commit_details.parquet")

In [2]:
import sys
sys.path.insert(0, '..')
from helpers import build_pr_states, categorize

# Closed without merge
rejected = prs[
    prs['merged_at'].isna() & prs['closed_at'].notna()
].copy()

pr_states = build_pr_states(reviews, rejected['id'])

rejected['review_category'] = rejected['id'].apply(lambda pr_id: categorize(pr_id, pr_states))

print(f"Closed-without-merge PRs (raw): {len(rejected):,}")
rejected['review_category'].value_counts()

Closed-without-merge PRs (raw): 7,270


review_category
no_reviews                                       5886
commented_only                                    975
changes_requested_no_approval                     209
approved_only                                     150
changes_requested_then_approved                    26
dismissed_no_approval                              14
dismissed_then_approved                             8
changes_requested_and_dismissed_then_approved       2
Name: count, dtype: int64

In [3]:
# Filter: Keep only PRs that had at least one review (exclude no_reviews)
rejected_prs = rejected[rejected['review_category'] != 'no_reviews'].copy()

print(f"Problematic-Rejected PRs (with reviews): {len(rejected_prs):,}")
rejected_prs['review_category'].value_counts()

Problematic-Rejected PRs (with reviews): 1,384


review_category
commented_only                                   975
changes_requested_no_approval                    209
approved_only                                    150
changes_requested_then_approved                   26
dismissed_no_approval                             14
dismissed_then_approved                            8
changes_requested_and_dismissed_then_approved      2
Name: count, dtype: int64

In [4]:
# Filter: Remove PRs closed within 60 seconds of opening (likely mistakes, not genuine rejections)
rejected_prs['created_at'] = pd.to_datetime(rejected_prs['created_at'], utc=True)
rejected_prs['closed_at']  = pd.to_datetime(rejected_prs['closed_at'],  utc=True)

open_duration_secs = (rejected_prs['closed_at'] - rejected_prs['created_at']).dt.total_seconds()

quick_closes = rejected_prs[open_duration_secs < 60].copy()
rejected_prs = rejected_prs[open_duration_secs >= 60].copy()

print(f"Removed {len(quick_closes):,} quick-close PRs (open < 1 min)")
print(f"After removing quick-close PRs: {len(rejected_prs):,}")

Removed 14 quick-close PRs (open < 1 min)
After removing quick-close PRs: 1,370


In [5]:
# Filter: Keep only PRs that touched fewer than 10 distinct files
# Derived from pr_commit_details — nunique() on filename so a file modified
# across multiple commits counts once
files_per_pr = (
    details.groupby("pr_id")["filename"]
    .nunique()
    .rename("n_files_changed")
)

rejected_prs = rejected_prs.merge(files_per_pr, left_on="id", right_index=True, how="left")
rejected_prs["n_files_changed"] = rejected_prs["n_files_changed"].fillna(0).astype(int)

before = len(rejected_prs)
rejected_prs = rejected_prs[rejected_prs["n_files_changed"] < 10].copy()

print(f"Removed {before - len(rejected_prs):,} PRs with >= 10 distinct files changed")
print(f"After n_files_changed < 10 filter: {len(rejected_prs):,}")
print()
print(rejected_prs["n_files_changed"].describe())

Removed 456 PRs with >= 10 distinct files changed
After n_files_changed < 10 filter: 914

count    914.000000
mean       3.665208
std        2.410172
min        0.000000
25%        2.000000
50%        3.000000
75%        5.000000
max        9.000000
Name: n_files_changed, dtype: float64


In [6]:
# Filter: Remove PRs where CHANGES_REQUESTED came only from bots
human_cr_pr_ids = set(
    reviews[
        (reviews['state'] == 'CHANGES_REQUESTED') &
        (reviews['user_type'] == 'User')
    ]['pr_id'].unique()
)

# Keep only PRs with at least one human CHANGES_REQUESTED
rejected_prs = rejected_prs[rejected_prs['id'].isin(human_cr_pr_ids)]
print(f"After requiring human CHANGES_REQUESTED: {len(rejected_prs):,}")

After requiring human CHANGES_REQUESTED: 139


In [7]:
pr_reviews_sub = reviews[
    reviews['pr_id'].isin(rejected_prs['id'])
][['pr_id', 'id', 'user', 'user_type', 'state', 'submitted_at']].copy()

pr_commits_sub = commits[
    commits['pr_id'].isin(rejected_prs['id'])
][['pr_id', 'sha', 'author', 'message']].copy()

print(f"Reviews (on rejected PRs): {len(pr_reviews_sub):,}")
print(f"Commits (on rejected PRs): {len(pr_commits_sub):,}")

Reviews (on rejected PRs): 622
Commits (on rejected PRs): 784


In [8]:
# Pull inline review comments from CHANGES_REQUESTED reviews
cr_review_ids = set(
    reviews[
        (reviews['pr_id'].isin(rejected_prs['id'])) &
        (reviews['state'] == 'CHANGES_REQUESTED') &
        (reviews['user_type'] == 'User')
    ]['id'].unique()
)

cr_inline_comments = rev_cmts[
    rev_cmts['pull_request_review_id'].isin(cr_review_ids)
][['pull_request_review_id', 'user', 'path', 'diff_hunk', 'body', 'created_at']].copy()

review_id_to_pr = reviews.set_index('id')['pr_id'].to_dict()
cr_inline_comments['pr_id'] = cr_inline_comments['pull_request_review_id'].map(review_id_to_pr)

print(f"PRs with inline CR comments:  {cr_inline_comments['pr_id'].nunique():,}")
print(f"Total inline CR comments:     {len(cr_inline_comments):,}")

PRs with inline CR comments:  80
Total inline CR comments:     326


In [9]:
print("=== Problematic-Rejected Clean Subset ===")
print(f"Total PRs:                      {len(rejected_prs):,}")
print()
print(rejected_prs['review_category'].value_counts().to_string())
print()
print(f"PRs with inline CR comments:    {cr_inline_comments['pr_id'].nunique():,}")

=== Problematic-Rejected Clean Subset ===
Total PRs:                      139

review_category
changes_requested_no_approval                    121
changes_requested_then_approved                   17
changes_requested_and_dismissed_then_approved      1

PRs with inline CR comments:    80


In [10]:
print(rejected_prs.columns.tolist())

['id', 'number', 'title', 'body', 'agent', 'user_id', 'user', 'state', 'created_at', 'closed_at', 'merged_at', 'repo_id', 'repo_url', 'html_url', 'review_category', 'n_files_changed']


In [11]:
# Spot-check: inspect a single PR
sample_id = rejected_prs.sample(10, random_state=42)['id'].iloc[4]
pr_html_url = prs.loc[prs['id'] == sample_id, 'html_url'].iloc[0]

print("=== PR ===")
print(prs[prs['id'] == sample_id][['id', 'title', 'agent', 'repo_url', 'html_url', 'created_at', 'closed_at']].to_string(index=False))

print("\n=== Reviews ===")
print(
    pr_reviews_sub[pr_reviews_sub['pr_id'] == sample_id]
    .sort_values('submitted_at')[['state', 'user', 'user_type', 'submitted_at']]
    .to_string(index=False)
)

print("\n=== Commits ===")
commits_sample = pr_commits_sub[pr_commits_sub['pr_id'] == sample_id][['sha', 'author', 'message']].copy()
commits_sample['url'] = pr_html_url + '/commits/' + commits_sample['sha']
print(commits_sample.to_string(index=False))

print("\n=== Inline CR Comments ===")
print(
    cr_inline_comments[cr_inline_comments['pr_id'] == sample_id]
    [['path', 'body']]
    .to_string(index=False)
)

=== PR ===
        id                                                             title   agent                                     repo_url                                      html_url           created_at            closed_at
3231720206 feat: Disable Optimize in no-db mode with fail-fast startup check Copilot https://api.github.com/repos/camunda/camunda https://github.com/camunda/camunda/pull/35372 2025-07-15T10:27:58Z 2025-07-24T01:27:23Z

=== Reviews ===
            state                   user user_type         submitted_at
CHANGES_REQUESTED               abremard      User 2025-07-16T04:10:02Z
        COMMENTED copilot-swe-agent[bot]       Bot 2025-07-16T04:24:59Z
        COMMENTED copilot-swe-agent[bot]       Bot 2025-07-16T04:25:07Z
        COMMENTED copilot-swe-agent[bot]       Bot 2025-07-16T04:25:16Z
CHANGES_REQUESTED               abremard      User 2025-07-16T06:50:06Z
        COMMENTED copilot-swe-agent[bot]       Bot 2025-07-16T07:00:56Z
        COMMENTED copilot-swe-ag

In [12]:

# Export filtered problematic-rejected PRs to CSV
output = rejected_prs[['id', 'title', 'repo_url']].rename(columns={
    'id': 'pr_id',
    'title': 'pr_name',
    'repo_url': 'repo_url',
})

output.to_csv('problematic_rejected_prs.csv', index=False)
print(f"Saved {len(output):,} rows to problematic_rejected_prs.csv")
output.head()


Saved 139 rows to problematic_rejected_prs.csv


,pr_id,pr_name,repo_url
58,2876006908,Improve list and collection materializers perf...,https://api.github.com/repos/zenml-io/zenml
365,2959025892,[fix][cli] Enhance split-bundle command to acc...,https://api.github.com/repos/apache/pulsar
552,3234115546,Fix Resource Group option sharing static state...,https://api.github.com/repos/Azure/azure-mcp
587,3081816409,Fix: Custom tags defined as floats being treat...,https://api.github.com/repos/navidrome/navidrome
643,3167462922,Avoid Pods crashlooping when elasticsearch is ...,https://api.github.com/repos/camunda/camunda
